## Analysis of the processed dataset

In [51]:
import sys
import os
import numpy as np
import pandas as pd
import s3fs
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from dotenv import load_dotenv
project_root = '/home/onyxia/work/election_modeling_uhcp'
sys.path.insert(0, project_root)
from src.components.data_processing.data_loader import DataLoader
load_dotenv()

True

In [52]:
os.environ["AWS_ACCESS_KEY_ID"] = 'LUEQJQVXXVT45IX81NRK'
os.environ["AWS_SECRET_ACCESS_KEY"] = 'kRA0FrBvUoHksj+u8V6DOxYvDwHTWGSA5MYr3QKz'
os.environ["AWS_SESSION_TOKEN"] = 'eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhY2Nlc3NLZXkiOiJMVUVRSlFWWFhWVDQ1SVg4MU5SSyIsImFsbG93ZWQtb3JpZ2lucyI6WyIqIl0sImF1ZCI6WyJtaW5pby1kYXRhbm9kZSIsIm9ueXhpYSIsImFjY291bnQiXSwiYXV0aF90aW1lIjoxNzY5NDQ2OTg4LCJhenAiOiJvbnl4aWEiLCJlbWFpbCI6ImFydGh1ci5tYW5jZWF1QHN0dWRlbnQtY3MuZnIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZXhwIjoxNzcwNTY1NjgzLCJmYW1pbHlfbmFtZSI6Ik1hbmNlYXUiLCJnaXZlbl9uYW1lIjoiQXJ0aHVyIiwiZ3JvdXBzIjpbIlVTRVJfT05ZWElBIl0sImlhdCI6MTc2OTk2MDg4MiwiaXNzIjoiaHR0cHM6Ly9hdXRoLmxhYi5zc3BjbG91ZC5mci9hdXRoL3JlYWxtcy9zc3BjbG91ZCIsImp0aSI6Im9ucnRydDo2MTcxNDU0Ni0wNmM4LTM0ZjQtMmJlNC04OTJjMGJkYmM2NjgiLCJsb2NhbGUiOiJlbiIsIm5hbWUiOiJBcnRodXIgTWFuY2VhdSIsInBvbGljeSI6InN0c29ubHkiLCJwcmVmZXJyZWRfdXNlcm5hbWUiOiJhcnRodXJtYW5jZWF1IiwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNzcGNsb3VkIl19LCJyZXNvdXJjZV9hY2Nlc3MiOnsiYWNjb3VudCI6eyJyb2xlcyI6WyJtYW5hZ2UtYWNjb3VudCIsIm1hbmFnZS1hY2NvdW50LWxpbmtzIiwidmlldy1wcm9maWxlIl19fSwicm9sZXMiOlsib2ZmbGluZV9hY2Nlc3MiLCJ1bWFfYXV0aG9yaXphdGlvbiIsImRlZmF1bHQtcm9sZXMtc3NwY2xvdWQiXSwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBncm91cHMgZW1haWwiLCJzaWQiOiIxNGE1ZTU2Zi01ZjQyLTkzYjQtMzNjMy1lMGI2NmZkYzg2MmEiLCJzdWIiOiJiYzc3Yjk3YS1lN2VkLTQ4ZWMtYmQ4Zi1lNzBjYjZhMTYyZGIiLCJ0eXAiOiJCZWFyZXIifQ.JiIC6q8sO0yXN3uCOh13F9qcTJHQoeiKI1XxoUM40-csxqyi2F3uWJeqSA9iEGzWARxsYqrq98b6orxqOn0MBw'
os.environ["AWS_DEFAULT_REGION"] = 'us-east-1'
fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

In [ ]:
dataset = DataLoader().load_dataset("s3://arthurmanceau/election_modeling_uhcp/data/derived/processed/data_ppar_pvoteD_pvoteG_pvoteCG_pvoteCD_pvoteC_pvoteTD_pvoteTG_pvoteGCG_pvoteDCD_1958_presidentiel_legislative_20260201_185009.parquet", fs=fs)

In [ ]:
X = dataset[(dataset['annee']==2022)&(dataset['type']==0)]
assert not X['codecommune'].duplicated().any()

PB — with the previous 

In [ ]:
L = [col for col in X.columns if 'pvote' in col]
print(L)
n_blocs = 5+1+2+2
n_elec = 4
assert n_elec * n_blocs - 1 == len(L)
assert np.isclose(
    X[['pvotepvoteD', 'pvotepvoteG', 'pvotepvoteCG', 'pvotepvoteCD', 'pvotepvoteC']].sum(axis=1),
    1.00
).all()
assert np.isclose(
    X[['pvotepvoteTD', 'pvotepvoteTG']].sum(axis=1),
    1.00
).all()
assert np.isclose(
    X[['pvotepvoteDCD', 'pvotepvoteC', 'pvotepvoteGCG']].sum(axis=1),
    1.00
).all()

assert np.isclose(
    X[['pvotepreviouspreviouspvoteD', 'pvotepreviouspreviouspvoteG', 'pvotepreviouspreviouspvoteCG', 'pvotepreviouspreviouspvoteCD', 'pvotepreviouspreviouspvoteC']].dropna().sum(axis=1),
    1.00
).all()
assert np.isclose(
    X[['pvotepreviouspreviouspvoteTD', 'pvotepreviouspreviouspvoteTG']].dropna().sum(axis=1),
    1.00
).all()
assert np.isclose(
    X[['pvotepreviouspreviouspvoteDCD', 'pvotepreviouspreviouspvoteC', 'pvotepreviouspreviouspvoteGCG']].dropna().sum(axis=1),
    1.00
).all()

assert np.isclose(
    X[['pvotepreviouspvoteTD', 'pvotepreviouspvoteTG']].dropna().sum(axis=1),
    1.00
).all()
assert np.isclose(
    X[['pvotepreviouspvoteDCD', 'pvotepreviouspvoteC', 'pvotepreviouspvoteGCG']].dropna().sum(axis=1),
    1.00
).all()
assert np.isclose(
    X[['pvotepreviouspvoteD', 'pvotepreviouspvoteG', 'pvotepreviouspvoteCG', 'pvotepreviouspvoteCD', 'pvotepreviouspvoteC']].dropna().sum(axis=1),
    1.00
).all()

In [ ]:
X['inscrits'].sum() # 48 803 175 

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(X['inscrits'], kde=True, color='skyblue', bins=300)
plt.scatter(X['inscrits'], [0.01]*len(X), alpha=0.5, color='red')

top_n = 5
largest_indices = X['inscrits'].nlargest(top_n).index
for idx in largest_indices:
    plt.text(X['inscrits'][idx], 0.02, str(X['codecommune'][idx]), rotation=45, fontsize=9, color='blue')

plt.xscale('log')  # Logarithmic scale for x-axis
plt.xlabel('inscrits (log scale)')
plt.ylabel('Density')
plt.title('Histogram and Density of inscrits (log scale) with Largest Points Annotated')
plt.show()

In [ ]:
features = [col for col in X.columns if '/' in col]
family_dict = defaultdict(list)
for col in features:
    family = col.split('/')[0]
    family_dict[family].append(col)

In [ ]:
nan_cols = X.columns[X.isna().all()]
print(list(nan_cols))

In [ ]:
#dfc = DataLoader.load_dataset("s3://arthurmanceau/election_modeling_uhcp/data/derived/cache/dfc_cached.parquet")

In [ ]:
#dfc.loc[dfc['codecommune'] == '69123', 'agesexcommunes/poph60p2022']

In [ ]:
#dfc.columns

In [ ]:
#dfc.loc[dfc['codecommune'] == '76095', ['agesexcommunes/propf40592022', 'agesexcommunes/propf60p2022']]

In [ ]:
X.loc[X['codecommune']=='01019', ['agesexcommunes/propf4059', 'agesexcommunes/propf60p']]

In [ ]:
X.loc[X['cspcommunes/pcapi'].isna(), ['codecommune', 'inscrits', 'cspcommunes/pcapi']]

In [ ]:
X.loc[X["empfoncommunes/con"].isna(), ['codecommune', 'inscrits', 'empfoncommunes/con']]

In [ ]:
L = ['13201',
 '13202',
 '13203',
 '13204',
 '13205',
 '13206',
 '13207',
 '13208',
 '13209',
 '13210',
 '13211',
 '13212',
 '13213',
 '13214',
 '13215',
 '13216']
nan_cols_in_L = X.loc[X['codecommune'].isin(L)].isna().all()
cols_all_nan_in_L = X.columns[nan_cols_in_L]
L2 = set(cols_all_nan_in_L)

In [ ]:
nan_cols_in_L = dataset.loc[dataset['codecommune'].isin(L)].isna().all()
cols_all_nan_in_L = dataset.columns[nan_cols_in_L]
L1= set(cols_all_nan_in_L)

In [ ]:
Lm = [el + '2022' for el in list(L2)]

In [ ]:
lp = list(set(Lm).intersection(dfc.columns))
dfc.loc[dfc['codecommune']=='13055', lp]

In [ ]:

# Find locations of inf values
inf_mask = X.isin([np.inf, -np.inf])

# Show rows and columns with inf values
inf_locations = inf_mask.any(axis=1)
print(X[inf_locations])

In [ ]:
L2 - L1

In [ ]:
cn = []
for family, cols in family_dict.items():
    cols_ = list(set(cols)- set(nan_cols))
    cols_nan = (X[cols_].isna().astype(int).sum()>0).index.tolist()
    cn += cols_nan

In [ ]:
dataset[(dataset['annee']==2022)&(dataset['type']==0)]['agesexcommunes/pop'].sum()

In [ ]:
#assert dataset[(dataset['annee']==2022)&(dataset['type']==0)]['agesexcommunes/pop'].sum() == dataset[(dataset['annee']==2022)&(dataset['type']==1)]['agesexcommunes/pop'].sum()
dataset[(dataset['annee']==2017)&(dataset['type']==0)]['agesexcommunes/pop'].sum()

In [ ]:
assert X['agesexcommunes/pop'].sum() == X['agesexcommunes/popf'].sum()+X['agesexcommunes/poph'].sum()

In [ ]:
X[cn].median()

In [ ]:
for family, cols in family_dict.items():
    cols_ = list(set(cols)- set(nan_cols))
    corr = X[cols_].corr()
    corr = corr[corr>0.8]
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title(f'Correlation Matrix for family: {family}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot histogram for each family
for family, cols in family_dict.items():
    cols_ = list(set(cols)- set(nan_cols))
    X[cols].hist(figsize=(12, 8), bins=30, layout=(-1, 3), sharex=False, sharey=False)
    plt.suptitle(f'Histograms for family: {family}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
n_th = 36529
n = X['codecommune'].nunique()
print(n) 
print(f"{(n*100/n_th):1f}%")
print(X['dep'].nunique()) # France metropolitaine 


In [ ]:
N = dataset.isnull().mean(axis=0)

In [ ]:
N

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
dataset[dataset['annee']==2017]['cspcommunes/perindp'].hist(bins=30)

In [ ]:
dataset.columns.to_list()

In [ ]:
dataset[['annee', 'cspcommunes/perindp']].dropna()['annee'].unique()

In [ ]:
N[N<0.2].to_frame().T.columns.to_list()

In [ ]:
import numpy as np
import pandas as pd
data = np.load('../../data/raw/geo_data/distance_cities/distance_matrix.npz')

In [ ]:
m = pd.read_parquet('../../data/raw/geo_data/distance_cities/mapping.parquet')

In [ ]:
m[m.iloc[:,0]=="05039"].index[0]

In [ ]:
array1 = data['array']

In [ ]:
array1

In [ ]:
class ComposeDistanceMatrix:

    def __init__(self):
        self.distance_array = None
        self.i = None
        self.j = None
        self.N = None
    
    def _load_files(self):
        distance_npz = np.load('../../data/raw/geo_data/distance_cities/distance_matrix.npz')
        self.distance_array = distance_npz['array']

        i_npz =  np.load('../../data/raw/geo_data/distance_cities/i.npz')['arr_0']
        self.i = i_npz

        j_npz =  np.load('../../data/raw/geo_data/distance_cities/j.npz')['arr_0']
        self.j = j_npz

        m = pd.read_parquet('../../data/raw/geo_data/distance_cities/mapping.parquet')
        self.mapping = {value: idx for idx, value in m.iloc[:, 0].items()}

        self.N = len(i)

    def w(alpha, beta):
        pass

In [ ]:
dm = ComposeDistanceMatrix()
dm._load_files()

In [ ]:
import pandas as pd

In [ ]:
X = pd.read_parquet('../../data/polls/2022/polls_t1.parquet')

In [ ]:
X[['C', 'CD', 'CG', 'G', 'D']].mean()

In [54]:
X = pd.read_parquet('s3://arthurmanceau/election_modeling_uhcp/data/raw/elections/legislative/2022/leg2022_csv/leg2022comm.parquet', filesystem=fs)

In [84]:
X

,dep,nomdep,codecommune,nomcommune,inscrits,votants,exprimes,voixAUG,voixNUP,voixDVG,...,ppar,pparratio,perpar,pblancnul,pblancnulratio,pins,pinsratio,pabs,pblancsnuls,electeurs
0,01,AIN,01001,L'ABERGEMENT-CLÃMENCIAT,644,343,339.0,4,52,16,...,0.532609,1.086149,0.741754,0.006211,0.592583,1.000000,1.075965,0.467391,0.006211,644.00000
1,01,AIN,01002,L'ABERGEMENT-DE-VAREY,218,133,128.0,4,44,0,...,0.610092,1.244161,0.970394,0.022936,2.188207,1.000000,1.075965,0.389908,0.022936,218.00000
2,01,AIN,01004,AMBÃRIEU-EN-BUGEY,8844,4123,4012.0,66,1196,33,...,0.466192,0.950705,0.343821,0.012551,1.197427,0.894426,0.962372,0.533808,0.012551,9887.90530
3,01,AIN,01005,AMBÃRIEUX-EN-DOMBES,1299,636,622.0,5,100,24,...,0.489607,0.998457,0.485243,0.010778,1.028238,0.848230,0.912666,0.510393,0.010778,1531.42490
4,01,AIN,01006,AMBLÃON,101,63,59.0,3,13,0,...,0.623762,1.272039,0.980940,0.039604,3.778448,1.000000,1.075965,0.376238,0.039604,101.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34865,95,VAL-D'OISE,95676,VILLERS-EN-ARTHIES,396,231,228.0,4,32,0,...,0.583333,1.189592,0.933780,0.007576,0.722771,1.000000,1.075965,0.416667,0.007576,396.00000
34866,95,VAL-D'OISE,95678,VILLIERS-ADAM,606,361,357.0,3,66,4,...,0.595710,1.214831,0.955794,0.006601,0.629741,0.837821,0.901466,0.404290,0.006601,723.30511
34867,95,VAL-D'OISE,95680,VILLIERS-LE-BEL,12560,4322,4200.0,36,1934,443,...,0.344108,0.701740,0.018288,0.009713,0.926713,0.897944,0.966157,0.655892,0.009713,13987.50500
34868,95,VAL-D'OISE,95682,VILLIERS-LE-SEC,111,61,60.0,0,16,4,...,0.549550,1.120697,0.820638,0.009009,0.859512,0.620843,0.668005,0.450450,0.009009,178.78928


In [58]:
Y = pd.read_csv('../../data/raw/2024_pre_processed.csv')

/tmp/ipykernel_7129/431602615.py:1: DtypeWarning: Columns (0,2,26,35,44,53,62,71,73,74,75,76,78,79,80,82,83,84,85,87,88,89,91,92,93,94,96,97,98,100,101,102,103,105,106,107,109,110,111,112,114,115,116,118,119,120,121,123,124,125,127,128,129,130,132,133,134,136,137,138,139,141,142,143,145,146,147,148,150,151,152,154,155,156,157,159,160,161,163,164,165,166,168,169,172,173,174,175,177,178,181,182,183,184,186,187,190,191,192,193,195,196,199,200,201,202,204,205,206,208,209,210,211,213,214,217,218,219,220,222,223,226,227,228,229,231,232,235,236,237,238,240,241,242,244,245,246,247,249,250,253,254,255,256,258,259,262,263,264,265,267,268,271,272,273,274,276,277,278,280,281,282,283,285,286,289,290,291,292,294,295,298,299,300,301,303,304,307,308,309,310,312,313,316,317,318,319,321,322,325,326,327,328,330,331,334,335,336,337,339,340,343,344,345,346,348,349,352,353,354,355,357,358,361,362,363,364,366,367,370,371,372,373,375,376,379,380,381,382,384,385,388,389,390,391,393,394,397,398,399,400,402,403,

In [91]:
Y.iloc[:, 18:].iloc[0].to_csv('m.csv')

In [94]:
Y

,Code département,Libellé département,Code commune,Libellé commune,Inscrits,Votants,% Votants,Abstentions,% Abstentions,Exprimés,...,Elu 203,Numéro de panneau 204,Nuance candidat 204,Nom candidat 204,Prénom candidat 204,Sexe candidat 204,Voix 204,% Voix/inscrits 204,% Voix/exprimés 204,Elu 204
0,1,Ain,1001,L'Abergement-Clémenciat,662,492,"74,32%",170,"25,68%",476,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Ain,1002,L'Abergement-de-Varey,228,178,"78,07%",50,"21,93%",171,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Ain,1004,Ambérieu-en-Bugey,8744,6037,"69,04%",2707,"30,96%",5890,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Ain,1005,Ambérieux-en-Dombes,1337,960,"71,80%",377,"28,20%",941,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Ain,1006,Ambléon,98,68,"69,39%",30,"30,61%",65,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35227,ZZ,Français établis hors de France,ZZ235,Bahamas (Nassau),163,50,"30,67%",113,"69,33%",50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35228,ZZ,Français établis hors de France,ZZ236,Astana,93,45,"48,39%",48,"51,61%",44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35229,ZZ,Français établis hors de France,ZZ237,Mossoul,1,0,"0,00%",1,"100,00%",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35230,ZZ,Français établis hors de France,ZZ238,Florence,4113,1223,"29,73%",2890,"70,27%",1199,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np

# Suppose df is your DataFrame

# Find the number of candidates (N)
N = 204

# Build the long format DataFrame for each row
voix_cols = [f'Voix {i}' for i in range(1, N+1)]
nuance_cols = [f'Nuance candidat {i}' for i in range(1, N+1)]
pvoix_cols = [f'% Voix/exprimés {i}' for i in range(1, N+1)]

records = []
for idx, row in Y.iterrows():
    for i in range(1, N+1):
        nuance = row.get(f'Nuance candidat {i}', None)
        voix = row.get(f'Voix {i}', None)
        pvoix = row.get(f'% Voix/exprimés {i}', None)
        if pd.notna(nuance) and nuance != '':
            # Clean voix and pvoix
            try:
                voix = float(str(voix).replace(',', '.'))
            except:
                voix = np.nan
            try:
                pvoix = float(str(pvoix).replace(',', '.').replace('%', ''))
            except:
                pvoix = np.nan
            records.append({'row_id': idx, 'Nuance': nuance, 'Voix': voix, 'PVoix': pvoix})

df_long = pd.DataFrame(records)

# Group by row and Nuance
summary = df_long.groupby(['row_id', 'Nuance'], dropna=False).agg({'Voix': 'sum', 'PVoix': 'sum'}).reset_index()

# (Optional) Pivot to wide format if you want one row per polling station
summary_wide = summary.pivot(index='row_id', columns='Nuance', values=['Voix', 'PVoix'])

# To flatten the columns (optional)
summary_wide.columns = [f"{stat}_{nuance}" for stat, nuance in summary_wide.columns]
summary_wide = summary_wide.reset_index()

In [96]:
Y

,Code département,Libellé département,Code commune,Libellé commune,Inscrits,Votants,% Votants,Abstentions,% Abstentions,Exprimés,...,Elu 203,Numéro de panneau 204,Nuance candidat 204,Nom candidat 204,Prénom candidat 204,Sexe candidat 204,Voix 204,% Voix/inscrits 204,% Voix/exprimés 204,Elu 204
0,1,Ain,1001,L'Abergement-Clémenciat,662,492,"74,32%",170,"25,68%",476,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Ain,1002,L'Abergement-de-Varey,228,178,"78,07%",50,"21,93%",171,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Ain,1004,Ambérieu-en-Bugey,8744,6037,"69,04%",2707,"30,96%",5890,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Ain,1005,Ambérieux-en-Dombes,1337,960,"71,80%",377,"28,20%",941,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Ain,1006,Ambléon,98,68,"69,39%",30,"30,61%",65,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35227,ZZ,Français établis hors de France,ZZ235,Bahamas (Nassau),163,50,"30,67%",113,"69,33%",50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35228,ZZ,Français établis hors de France,ZZ236,Astana,93,45,"48,39%",48,"51,61%",44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35229,ZZ,Français établis hors de France,ZZ237,Mossoul,1,0,"0,00%",1,"100,00%",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35230,ZZ,Français établis hors de France,ZZ238,Florence,4113,1223,"29,73%",2890,"70,27%",1199,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
summary_wide

,row_id,Voix_COM,Voix_DIV,Voix_DSV,Voix_DVC,Voix_DVD,Voix_DVG,Voix_ECO,Voix_ENS,Voix_EXD,...,PVoix_LR,PVoix_RDG,PVoix_REC,PVoix_REG,PVoix_RN,PVoix_SOC,PVoix_UDI,PVoix_UG,PVoix_UXD,PVoix_VEC
0,0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,86.0,NaN,...,9.87,NaN,NaN,NaN,54.20,NaN,NaN,17.44,NaN,NaN
1,1,NaN,NaN,NaN,NaN,40.0,NaN,2.0,19.0,NaN,...,2.92,NaN,0.00,NaN,NaN,NaN,NaN,36.26,24.56,NaN
2,2,NaN,NaN,NaN,NaN,727.0,NaN,51.0,789.0,NaN,...,4.45,NaN,1.02,NaN,NaN,NaN,NaN,29.86,36.47,NaN
3,3,NaN,1.0,NaN,NaN,NaN,NaN,NaN,198.0,NaN,...,6.48,NaN,NaN,NaN,52.50,NaN,NaN,18.38,NaN,NaN
4,4,NaN,1.0,0.0,NaN,NaN,NaN,NaN,22.0,NaN,...,4.62,NaN,NaN,NaN,36.92,NaN,NaN,21.54,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35227,35227,NaN,0.0,NaN,2.0,NaN,NaN,NaN,17.0,NaN,...,14.00,NaN,18.00,NaN,22.00,NaN,NaN,8.00,NaN,NaN
35228,35228,NaN,2.0,4.0,NaN,NaN,0.0,4.0,12.0,3.0,...,NaN,NaN,2.27,0.0,15.91,NaN,NaN,25.00,NaN,NaN
35229,35229,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,0.00,NaN,0.00,NaN,NaN,NaN,NaN,0.00,0.00,NaN
35230,35230,NaN,37.0,NaN,61.0,13.0,28.0,NaN,313.0,NaN,...,14.18,NaN,4.25,NaN,NaN,NaN,NaN,43.87,NaN,NaN


2 Grouping of tendance

In [62]:
Y.columns

Index(['Code département', 'Libellé département', 'Code commune',
       'Libellé commune', 'Inscrits', 'Votants', '% Votants', 'Abstentions',
       '% Abstentions', 'Exprimés',
       ...
       'Elu 203', 'Numéro de panneau 204', 'Nuance candidat 204',
       'Nom candidat 204', 'Prénom candidat 204', 'Sexe candidat 204',
       'Voix 204', '% Voix/inscrits 204', '% Voix/exprimés 204', 'Elu 204'],
      dtype='object', length=1854)

In [ ]:
rename_cols = {
    "Code département":'dep',
    "Libellé département":'nomdep',
    'Code Commune':'codecommune',
    "Libellé commune" :'nomcommune',
    'Inscrits':'inscrits',
    "Votants" : "votants",
    "Exprimés" : "exprimes"
}

In [79]:
nuance = set(Y['Nuance candidat 1'].unique().tolist())
print(len(nuance))
nuance = nuance.union(set(Y['Nuance candidat 2'].unique().tolist()))
print(len(nuance))
nuance = nuance.union(set(Y['Nuance candidat 3'].unique().tolist()))
print(len(nuance))
nuance = nuance.union(set(Y['Nuance candidat 4'].unique().tolist()))
print(len(nuance))
nuance = nuance.union(set(Y['Nuance candidat 5'].unique().tolist()))
print(len(nuance))
nuance = nuance.union(set(Y['Nuance candidat 6'].unique().tolist()))
print(len(nuance))

18
19
21
21
22
22


In [80]:
nuance

{'COM',
 'DIV',
 'DSV',
 'DVC',
 'DVD',
 'DVG',
 'ECO',
 'ENS',
 'EXD',
 'EXG',
 'FI',
 'HOR',
 'LR',
 'RDG',
 'REC',
 'REG',
 'RN',
 'SOC',
 'UDI',
 'UG',
 'UXD',
 nan}

In [83]:
X[X['plm']==1]

,dep,nomdep,codecommune,nomcommune,inscrits,votants,exprimes,voixAUG,voixNUP,voixDVG,...,ppar,pparratio,perpar,pblancnul,pblancnulratio,pins,pinsratio,pabs,pblancsnuls,electeurs
4412,13,BOUCHES-DU-RHONE,13201,MARSEILLE,20669,8999,8854.000000,112,5540,0,...,0.435386,0.887884,0.198390,0.007015,0.669304,NaN,NaN,0.564614,0.007015,NaN
4413,13,BOUCHES-DU-RHONE,13202,MARSEILLE,12830,4899,4808.000000,90,2172,0,...,0.381839,0.778686,0.051493,0.007093,0.676690,NaN,NaN,0.618161,0.007093,NaN
4414,13,BOUCHES-DU-RHONE,13203,MARSEILLE,21680,6096,5944.000000,153,2846,0,...,0.281181,0.573412,0.000867,0.007011,0.668897,NaN,NaN,0.718819,0.007011,NaN
4415,13,BOUCHES-DU-RHONE,13204,MARSEILLE,30131,13320,13110.000000,192,5407,23,...,0.442070,0.901513,0.225920,0.006970,0.664937,NaN,NaN,0.557930,0.006970,NaN
4416,13,BOUCHES-DU-RHONE,13205,MARSEILLE,26279,12717,12496.000000,158,5983,22,...,0.483923,0.986864,0.450295,0.008410,0.802340,NaN,NaN,0.516077,0.008410,NaN
4417,13,BOUCHES-DU-RHONE,13206,MARSEILLE,26947,13219,13031.000000,95,5419,15,...,0.490556,1.000390,0.492035,0.006977,0.665614,NaN,NaN,0.509444,0.006977,NaN
4418,13,BOUCHES-DU-RHONE,13207,MARSEILLE,26326,13348,13149.000000,81,3891,0,...,0.507027,1.033981,0.593401,0.007559,0.721179,NaN,NaN,0.492973,0.007559,NaN
4419,13,BOUCHES-DU-RHONE,13208,MARSEILLE,57287,27503,27136.000000,137,6210,0,...,0.480091,0.979051,0.429628,0.006406,0.611202,NaN,NaN,0.519909,0.006406,NaN
4420,13,BOUCHES-DU-RHONE,13209,MARSEILLE,49947,22378,22115.000000,97,4782,539,...,0.448035,0.913678,0.254551,0.005266,0.502367,NaN,NaN,0.551965,0.005266,NaN
4421,13,BOUCHES-DU-RHONE,13210,MARSEILLE,34533,13189,12968.000000,89,2994,353,...,0.381925,0.778859,0.051493,0.006400,0.610566,NaN,NaN,0.618075,0.006400,NaN
